# 00 — Entegrasyon kontrolü (Adım 0)
Önbelleği Data Conventions Final'e göre okur, sekiz doğrulayıcıyı çalıştırır, sayıları yazdırır. Ağa çıkmaz (önbellek önce `python -m src.build fafb` ile kurulmalı).

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
import config
from src import validate as V
from src.build import build_fafb
print("Kök klasör:", ROOT)

Kök klasör: C:\connectome


In [2]:
n, e, roi, ann_raw = build_fafb()   # dosyalar varsa sadece diskten okur

[cache] fafb_annotations_raw_783_ann3.2.0.parquet: 139,248 satır diskten (0.7 sn)


[cache] fafb_edgeroi_783_buhmann_min1.parquet: 16,847,997 satır diskten (3.3 sn)


[cache] fafb_edges_783_buhmann_min1.parquet: 15,091,983 satır diskten (1.8 sn)


[cache] fafb_neurons_783_ann3.2.0.parquet: 139,262 satır diskten (0.2 sn)


Doğrulama: fafb neurons/edges/edgeroi tüm kontroller geçti (References dahil)


ADIM0 fafb nöron: 139,262


ADIM0 fafb kenar >= 1: 15,091,983


ADIM0 fafb kenar >= 5: 2,700,513


ADIM0 fafb kenar >= 10: 1,066,822


ADIM0 fafb toplam sinaps (>=1): 54,492,922


## Doğrulayıcılar

In [3]:
V.validate_neurons(n)
V.validate_edges(e, n)
V.validate_edgeroi(roi, n)
V.check_plausibility(n, e)          # FAFB'de n_post boş, bu yüzden 0 nöron karşılaştırılır
V.assert_interchangeable(n, n)      # MaleCNS gelince: V.assert_interchangeable(n, n_male)
print("Tüm kontroller geçti")

Tüm kontroller geçti


## Adım 0 sayıları
Beklenen: nöron 139,248 + eksik kimlikler (7), kenar ≥5 = 2,700,513, öz-bağlantı 0.

In [4]:
rows = {"nöron": len(n), "öz-bağlantı": int((e.pre_id == e.post_id).sum())}
for t in (1, 5, 10, 31):
    rows[f"kenar >= {t}"] = int((e.syn_count >= t).sum())
pd.Series(rows).map("{:,}".format)

nöron             139,262
öz-bağlantı             0
kenar >= 1     15,091,983
kenar >= 5      2,700,513
kenar >= 10     1,066,822
kenar >= 31       165,352
dtype: object

## README için kaynak kolonlar ve değer sayımları

In [5]:
print(list(ann_raw.columns))
for col in ["super_class", "side", "top_nt", "status", "dimorphism"]:
    print("\n", col); print(ann_raw[col].value_counts(dropna=False).head(15).to_string())

['supervoxel_id', 'root_id', 'pos_x', 'pos_y', 'pos_z', 'soma_x', 'soma_y', 'soma_z', 'nucleus_id', 'flow', 'super_class', 'cell_class', 'cell_sub_class', 'supertype', 'cell_type', 'hemibrain_type', 'ito_lee_hemilineage', 'hartenstein_hemilineage', 'top_nt', 'top_nt_conf', 'known_nt', 'known_nt_source', 'side', 'nerve', 'vfb_id', 'fbbt_id', 'status', 'dimorphism', 'matching_notes', 'fru_dsx', 'synonyms']

 super_class
super_class
optic                 77541
central               32383
sensory               16907
visual_projection      8038
ascending              1750
descending             1303
sensory_ascending       612
visual_centrifugal      524
motor                   110
endocrine                80

 side
side
left      69956
right     69088
center      174
na           30

 top_nt
top_nt
acetylcholine    86193
glutamate        24875
gaba             19171
dopamine          5909
serotonin         2282
None               602
octopamine         216

 status
status
None           13

In [6]:
for col in ["super_class", "side", "nt_type", "region"]:
    print("\n", col); print(n[col].value_counts(dropna=False).head(15).to_string())
n.dtypes


 super_class
super_class
optic                 77541
central               32383
sensory               16907
visual_projection      8038
ascending              2362
descending             1303
visual_centrifugal      524
motor                   110
endocrine                80
unknown                  14

 side
side
left       69956
right      69088
center       174
unknown       44

 nt_type
nt_type
acetylcholine    86193
glutamate        24875
gaba             19171
dopamine          5909
serotonin         2282
unknown            616
octopamine         216
histamine            0

 region
region
ME_L       30821
ME_R       30526
LO_L        8870
LO_R        6193
LOP_R       6106
GNG         5860
LA_L        4830
LA_R        4292
LOP_L       3112
AVLP_R      2087
FB          1893
SMP_R       1882
MB_ML_L     1813
SMP_L       1786
AVLP_L      1780


neuron_id                  int64
dataset                 category
release           string[python]
cell_type         string[python]
cell_class        string[python]
cell_sub_class    string[python]
super_class             category
side                    category
region            string[python]
nt_type                 category
nt_confidence            float32
n_pre                      Int32
n_post                     Int32
flywire_type      string[python]
manc_type         string[python]
hemibrain_type    string[python]
dimorphism        string[python]
status            string[python]
hemilineage       string[python]
nt_known          string[python]
dtype: object